# Algoritam šišmiša za optimizaciju hiperparametara

Dataset: **Breast Cancer**

Model: **SVC**

Ovdje konkretno koristimo bat algoritam za optimizaciju C (regularizacijski
parametar) parametra SVM-a.

In [104]:
import pandas as pd
import numpy as np

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.datasets import load_breast_cancer

In [105]:
# Ucitavanje dataseta i podjela na trening i test skupove
df = load_breast_cancer()
X = df.data
y = df.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

In [106]:
def objective(C):
    """
    Trenira SVM sa datim C i vraca negativni F1 score.

    Args:
        C (float): Regularizacijski parametar SVM-a.

    Returns:
        float: Negativni F1 score na test skupu.
    """
    model = SVC(C=C)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return -f1_score(y_test, y_pred, average='weighted')


def popravi_granice(C):
    """
    Vraca C unutar dozvoljenog raspona.

    Args:
        C (float): Regularizacijski parametar SVM-a.
    
    Returns:
        float: C unutar dozvoljenog raspona.
    """
    return np.clip(C, 0.001, 100.0)

In [107]:
# Parametri
n_bats = 20
n_iter = 50
f_min = 0.0
f_max = 2.0
alpha = 0.9
gamma_ba = 0.9

# Inicijalizacija
np.random.seed(1)

# pozicije (C vrijednosti)
x = np.random.uniform(0.001, 100.0, size=n_bats)
# brzine
v = np.zeros(n_bats)
# frekvencije
f = np.zeros(n_bats)
# glasnoca
loudness = np.ones(n_bats)
# pulse rate
pulse_rate = np.zeros(n_bats)
r0 = 1.0

# Evaluacija pocetnih pozicija
fitness = np.array([objective(x[i]) for i in range(n_bats)])

# Globalno najbolje
best_idx = np.argmin(fitness)
best = x[best_idx].copy()
best_fitness = fitness[best_idx]

print(f"Inicijalno: C = {best:.4f}, F1 = {-best_fitness:.4f}")
print("=" * 50)

# Glavna petlja
for t in range(1, n_iter + 1):
    for i in range(n_bats):
        # beta = random(0, 1)
        beta = np.random.uniform(0, 1)

        # f[i] = f_min + (f_max - f_min) * beta
        f[i] = f_min + (f_max - f_min) * beta

        # v[i] = v[i] + (x[i] - best) * f[i]
        v[i] = v[i] + (x[i] - best) * f[i]

        # x_new = x[i] + v[i]
        x_new = x[i] + v[i]

        # ako random > pulse_rate: lokalna pretraga
        if np.random.random() > pulse_rate[i]:
            epsilon = np.random.uniform(-1, 1)
            avg_loudness = np.mean(loudness)
            x_new = best + epsilon * avg_loudness

        # popravi granice
        x_new = popravi_granice(x_new)

        # evaluiraj
        fitness_new = objective(x_new)

        # prihvati ako je bolje I random < loudness
        if fitness_new < fitness[i] and np.random.random() < loudness[i]:
            x[i] = x_new
            fitness[i] = fitness_new
            loudness[i] = alpha * loudness[i]
            pulse_rate[i] = r0 * (1 - np.exp(-gamma_ba * t))

        # azuriraj globalno najbolje
        if fitness[i] < best_fitness:
            best = x[i].copy()
            best_fitness = fitness[i]

    if t % 10 == 0 or t == 1:
        prosjecni_C = np.mean(x)
        print(f"Iteracija {t:3d}: Najbolji C = {best:.4f} | Prosjek svih šišmiša (C) = {prosjecni_C:.4f} | F1 = {-best_fitness:.4f}")

print("=" * 50)
print(f"\nNajbolji C = {best:.4f}, F1 = {-best_fitness:.4f}")

Inicijalno: C = 41.7028, F1 = 0.9472
Iteracija   1: Najbolji C = 41.7028 | Prosjek svih šišmiša (C) = 42.3534 | F1 = 0.9472
Iteracija  10: Najbolji C = 41.7028 | Prosjek svih šišmiša (C) = 42.3534 | F1 = 0.9472
Iteracija  20: Najbolji C = 41.7028 | Prosjek svih šišmiša (C) = 42.3534 | F1 = 0.9472
Iteracija  30: Najbolji C = 41.7028 | Prosjek svih šišmiša (C) = 42.3534 | F1 = 0.9472
Iteracija  40: Najbolji C = 3.6254 | Prosjek svih šišmiša (C) = 9.2490 | F1 = 0.9559
Iteracija  50: Najbolji C = 3.6254 | Prosjek svih šišmiša (C) = 3.3709 | F1 = 0.9559

Najbolji C = 3.6254, F1 = 0.9559


In [108]:
# Evaluacija finalnog modela sa C parametrom koji je odabrao bat algoritam
final_model = SVC(C=best)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
f1_score(y_test, y_pred, average='weighted')

0.9558976047642971

In [109]:
# Evaluacija modela sa "nasumicnim" C parametrom
random_C_model = SVC(C=50.5)
random_C_model.fit(X_train, y_train)
y_random_C_pred = random_C_model.predict(X_test)
f1_score(y_test, y_random_C_pred, average='weighted')

0.9471833355767937